In [0]:
%sql

create connection if not exists us_earthquakes_connection
type HTTP
options(
  host = 'https://earthquake.usgs.gov',
  port = 443,
  base_path = '/earthquakes/feed/v1.0/',
  bearer_token = 'na'
)

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

connection = w.connections.get("us_earthquakes_connection")

print(connection)

In [0]:
base_url = f"{connection.options['host']}{connection.options['base_path']}"
url = f"{base_url}summary/all_day.geojson"
print(url)

In [0]:
dbutils.widgets.text("catlog_name","databricks_as","catlog")

catalog_name = dbutils.widgets.get("catlog_name")

print(catalog_name)

In [0]:
spark.sql(f"use catalog {catalog_name}")

spark.sql("use schema bronze")

#spark.sql("create volume if not exists earthquake_data")

In [0]:
import requests
import json
import datetime


response = requests.get(url)

if response.status_code != 200:
    raise Exception(f"Failed to retrieve data from {url}. Status code: {response.status_code}")

data= response.json()
current_date = datetime.datetime.now().strftime("%Y-%m-%d")
dbutils.fs.put(f"/Volumes/{catalog_name}/bronze/earthquake_data/earthquake_data_{current_date}.json", json.dumps(data), overwrite=True)